# S13: Ingeniería de características para NovaMarket

Este catálogo construye variables derivadas, agregaciones históricas por cliente y un score de dominio RFM para apoyar análisis de retención y valor. Los agregados de cada pedido usan únicamente compras de días anteriores; las compras del mismo día se agrupan para que ninguna se adelante artificialmente a otra.

La salida conserva las columnas de S12. `fuga_cliente` es una etiqueta y no participa en ninguna fórmula; debe excluirse de las variables predictoras al entrenar un modelo.

## 1. Lectura y validación de la entrada

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ARCHIVO_ENTRADA = 'S12_PD_PallaresArbelaezSanchez_DatasetIntegrado.csv'
WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / 'S12' / ARCHIVO_ENTRADA).exists():
    WORKSPACE = WORKSPACE.parent
S13_DIR = WORKSPACE / 'S13'
S13_DIR.mkdir(exist_ok=True)
ruta_entrada = WORKSPACE / 'S12' / ARCHIVO_ENTRADA

df = pd.read_csv(
    ruta_entrada,
    encoding='utf-8-sig',
    dtype={'id_pedido': 'string', 'id_cliente': 'string'},
)
columnas_requeridas = {
    'id_pedido', 'id_cliente', 'fecha_pedido', 'monto_compra',
    'unidades_vendidas', 'fecha_inscripcion_club'
}
faltantes = columnas_requeridas - set(df.columns)
assert not faltantes, f'Faltan columnas requeridas: {sorted(faltantes)}'
assert df['id_pedido'].is_unique, 'S12 debe conservar una fila por pedido.'
filas_entrada = len(df)
df['fecha_pedido'] = pd.to_datetime(df['fecha_pedido'], errors='coerce')
df['fecha_inscripcion_club'] = pd.to_datetime(
    df['fecha_inscripcion_club'], errors='coerce'
)
df['monto_compra'] = pd.to_numeric(df['monto_compra'], errors='coerce')
df['unidades_vendidas'] = pd.to_numeric(df['unidades_vendidas'], errors='coerce')
print(f'Pedidos leídos: {filas_entrada:,}')
print(f'Fechas de pedido no interpretables: {df["fecha_pedido"].isna().sum():,}')

## 2. Variables derivadas

Se calculan con datos del pedido y del CRM disponibles en esa fila. Una antigüedad negativa se considera inválida y queda nula; el monto por unidad solo se define si las unidades son positivas.

In [ ]:
df['antiguedad_cliente_dias'] = (
    df['fecha_pedido'] - df['fecha_inscripcion_club']
).dt.days
df['antiguedad_cliente_dias'] = df['antiguedad_cliente_dias'].where(
    df['antiguedad_cliente_dias'].ge(0)
)
df['monto_por_unidad'] = (
    df['monto_compra'] / df['unidades_vendidas'].where(df['unidades_vendidas'].gt(0))
)
df[['antiguedad_cliente_dias', 'monto_por_unidad']].describe()

## 3. Agregaciones históricas y feature de dominio

Las agregaciones se calculan por cliente y fecha. Primero se consolida cada día; luego se resta el día actual de los acumulados. Así, todas las filas del mismo cliente y día reciben el mismo historial anterior y no incorporan compras que aún no habían ocurrido al inicio de ese día. El RFM usa umbrales fijos documentados en el diccionario, no cuantiles calculados con todo el periodo.

In [ ]:
columnas_agregadas = [
    'pedidos_previos_cliente', 'gasto_previo_cliente',
    'ticket_promedio_previo_cliente', 'dias_desde_compra_previa'
]
df[columnas_agregadas] = np.nan
mascara_historial = df['id_cliente'].notna() & df['fecha_pedido'].notna()
validos = df.loc[
    mascara_historial,
    ['id_cliente', 'id_pedido', 'fecha_pedido', 'monto_compra']
].copy()
validos['_fila_origen'] = validos.index

diario = (
    validos.groupby(['id_cliente', 'fecha_pedido'], as_index=False, sort=True)
    .agg(
        pedidos_dia=('id_pedido', 'nunique'),
        monto_dia=('monto_compra', 'sum'),
    )
    .sort_values(['id_cliente', 'fecha_pedido'], kind='stable')
)
por_cliente = diario.groupby('id_cliente', sort=False)
diario['pedidos_previos_cliente'] = (
    por_cliente['pedidos_dia'].cumsum() - diario['pedidos_dia']
)
diario['gasto_previo_cliente'] = (
    por_cliente['monto_dia'].cumsum() - diario['monto_dia']
)
diario['fecha_compra_previa'] = por_cliente['fecha_pedido'].shift(1)
diario['dias_desde_compra_previa'] = (
    diario['fecha_pedido'] - diario['fecha_compra_previa']
).dt.days
diario['ticket_promedio_previo_cliente'] = (
    diario['gasto_previo_cliente'] / diario['pedidos_previos_cliente'].where(
        diario['pedidos_previos_cliente'].gt(0)
    )
)

enriquecidas = validos.merge(
    diario[['id_cliente', 'fecha_pedido', *columnas_agregadas]],
    on=['id_cliente', 'fecha_pedido'],
    how='left',
    validate='many_to_one',
)
enriquecidas = enriquecidas.set_index('_fila_origen')
df.loc[enriquecidas.index, columnas_agregadas] = enriquecidas[columnas_agregadas]
df['pedidos_previos_cliente'] = df['pedidos_previos_cliente'].astype('Int64')

dias_previos = df['dias_desde_compra_previa']
frecuencia_previa = df['pedidos_previos_cliente']
gasto_previo = df['gasto_previo_cliente']
condiciones_recencia = [
    dias_previos.isna().to_numpy(dtype=bool)
    | dias_previos.le(30).fillna(False).to_numpy(dtype=bool),
    dias_previos.le(90).fillna(False).to_numpy(dtype=bool),
]
condiciones_frecuencia = [
    frecuencia_previa.ge(5).fillna(False).to_numpy(dtype=bool),
    frecuencia_previa.ge(2).fillna(False).to_numpy(dtype=bool),
]
condiciones_monetarias = [
    gasto_previo.ge(1_000_000).fillna(False).to_numpy(dtype=bool),
    gasto_previo.ge(250_000).fillna(False).to_numpy(dtype=bool),
]
score_recencia = np.select(condiciones_recencia, [3, 2], default=1)
score_frecuencia = np.select(condiciones_frecuencia, [3, 2], default=1)
score_monetario = np.select(condiciones_monetarias, [3, 2], default=1)
df['rfm_score'] = pd.Series(
    score_recencia + score_frecuencia + score_monetario, index=df.index
).where(frecuencia_previa.notna())

print(f'Clientes con historial utilizable: {validos["id_cliente"].nunique():,}')
df[['id_pedido', 'id_cliente', 'fecha_pedido', *columnas_agregadas, 'rfm_score']].head()

## 4. Controles de consistencia y exportación

In [ ]:
primer_dia_cliente = diario.groupby('id_cliente', sort=False).head(1)
assert primer_dia_cliente['pedidos_previos_cliente'].eq(0).all()
assert primer_dia_cliente['gasto_previo_cliente'].eq(0).all()
assert len(df) == filas_entrada, 'La ingeniería cambió el número de filas.'
assert df['id_pedido'].is_unique, 'Se duplicaron pedidos.'
assert df['rfm_score'].dropna().between(3, 9).all()
assert df['fuga_cliente'].equals(
    pd.read_csv(ruta_entrada, encoding='utf-8-sig', usecols=['fuga_cliente'])['fuga_cliente']
), 'La etiqueta de salida fue modificada.'

columnas_features = [
    'antiguedad_cliente_dias', 'monto_por_unidad', *columnas_agregadas, 'rfm_score'
]
print('Filas conservadas:', f'{len(df):,}')
print('Features nuevas:', ', '.join(columnas_features))
print('RFM nulo por falta de cliente/fecha:', f'{df["rfm_score"].isna().sum():,}')
display(df[columnas_features].describe().T)

ruta_salida = S13_DIR / 'S13_dataset_con_features.csv'
df.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f'Exportado: {ruta_salida}')